# OPT-In Family Services Data Quality and Outcomes Analysis

This notebook accompanies the scripted pipeline in `src/`. It uses only synthetic data and is intended as a reviewer-friendly walkthrough of the cleaning, validation, KPI, and visualization workflow.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'optin_referrals_raw.csv'
CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'optin_referrals_clean.csv'
QUALITY_PATH = PROJECT_ROOT / 'data' / 'processed' / 'data_quality_summary.json'
KPI_PATH = PROJECT_ROOT / 'data' / 'processed' / 'kpi_summary.json'
VISUALS_DIR = PROJECT_ROOT / 'visuals'

## Load Raw and Cleaned Data

In [ ]:
raw = pd.read_csv(RAW_PATH)
clean = pd.read_csv(CLEAN_PATH)

print(f'Raw rows: {len(raw):,}')
print(f'Cleaned rows: {len(clean):,}')
clean.head()

## Data Quality Summary

In [ ]:
quality = json.loads(QUALITY_PATH.read_text())
pd.DataFrame(
    {
        'metric': [
            'Input rows',
            'Output rows',
            'Duplicate family IDs detected',
            'Invalid referral dates removed',
            'Fund outliers capped',
            'Records requiring case review'
        ],
        'value': [
            quality['input_rows'],
            quality['output_rows'],
            quality['duplicate_family_ids_detected'],
            quality['rows_removed_invalid_dates'],
            quality['records_with_capped_fund_amounts'],
            quality['records_requiring_case_review']
        ]
    }
)

## KPI Summary

In [ ]:
kpis = json.loads(KPI_PATH.read_text())
pd.DataFrame(
    {
        'KPI': [
            'Total referrals',
            'Engagement rate',
            'Outreach completion rate',
            'Follow-up completion rate',
            'Average outreach attempts',
            'Total flexible funds'
        ],
        'Value': [
            kpis['total_referrals'],
            f"{kpis['engagement_rate']:.1%}",
            f"{kpis['outreach_completion_rate']:.1%}",
            f"{kpis['follow_up_completion_rate']:.1%}",
            kpis['average_outreach_attempts'],
            f"${kpis['total_flexible_funds']:,.0f}"
        ]
    }
)

## Site and Service Views

In [ ]:
engagement_by_site = (
    clean.assign(engaged=clean['engagement_status'].eq('Engaged'))
    .groupby('site_state')['engaged']
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)
engagement_by_site

In [ ]:
clean['service_type'].value_counts()

## Generated Visuals

The scripted pipeline creates the following charts for inclusion in a stakeholder dashboard or report:

- `monthly_referral_volume.png`
- `engagement_rate_by_site.png`
- `flexible_funds_by_site.png`
- `service_type_breakdown.png`
- `outcome_status_by_site.png`